In [ ]:
from utils_sm import *
from utils_SDEMEM_realdata import *
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import torch
import math
import random
import matplotlib.pyplot as plt
import torch.optim as optim
from tqdm import tqdm
import time

In [ ]:
obs_size = 40

In [ ]:
# ========== load observed data
df = pd.read_excel("realdata/20160427_mean_eGFP.xlsx", header=None)
x_obs = torch.tensor(df.to_numpy(), dtype=torch.float32)[:, 1:].T.log() 

x_obs = x_obs[:obs_size]

In [ ]:
# ===== Setting for real data ===== #
var_name = ['log_m0', 'log_scale', 'log_offset', 'log_sigma', 'mu_delta', 'mu_gamma', 'mu_k', 'mu_t0', 'log_tau_delta', 'log_tau_gamma', 'log_tau_k', 'log_tau_t0']

# The first 8 parameters have normal priors
prior_mean = torch.tensor([5, 1, 3, -1, -1, -5, 0.5, 0], dtype = torch.float32).unsqueeze(0).to(device)
prior_std = torch.tensor([1, 1, 1, 1, 1, 2, 1, 1], dtype = torch.float32).unsqueeze(0).to(device)

# For the last 4 parameters, the precision follows Gamma priors
prior_alpha = torch.tensor([2, 2, 2, 2], dtype = torch.float32).unsqueeze(0).to(device)
prior_beta = torch.tensor([0.5, 0.5, 0.5, 0.5], dtype = torch.float32).unsqueeze(0).to(device)

T = 30
theta_dim = 12
x_dim = 180 

In [ ]:
# ===== Load results ===== #
theta_ours = torch.from_numpy(np.load(f"sample_res_all/theta_post_precond_mchain_sameprior_task{0}.npy"))
theta_NPE = torch.from_numpy(np.load(f"sample_res_all/theta_post_NPE_newdeepsets_task{0}.npy")) 

In [ ]:
theta_set_ABC_all = []
W1_set_all = []
for i in range(10):
    theta_set_ABC_all.append(np.load(f'res_ABC/theta_set_batch{i}.npy'))
    W1_set_all.append(np.load(f'res_ABC/W1_set_batch{i}.npy'))

theta_set_ABC = torch.from_numpy(np.concatenate(theta_set_ABC_all))
W1_set_ABC = torch.from_numpy(np.concatenate(W1_set_all))

# W1_set_ABC contains some exact zero. This is because ot.emd2 returns 0 when there is nan in the data. So we exclude those.
nonzero_idx = torch.where(W1_set_ABC != 0)[0]
W1_set_ABC = W1_set_ABC[nonzero_idx]
theta_set_ABC = theta_set_ABC[nonzero_idx]

smallest_values, smallest_indices = torch.topk(W1_set_ABC, 1000, largest=False) 
theta_ABC = theta_set_ABC[smallest_indices].clone()

In [ ]:
def normalized_weight(log_w):
    tmp_w = (log_w - log_w.max()).exp()
    tmp_w = tmp_w / tmp_w.sum()

    thresh_id = 0
    while tmp_w.max() > 100 / log_w.shape[0]:
        thresh_id += 1
        clip = log_w.sort(descending=True).values[thresh_id]
        log_w = torch.clamp(log_w, max = clip)
        tmp_w = (log_w - log_w.max()).exp()
        tmp_w = tmp_w / tmp_w.sum()
    return tmp_w

In [ ]:
# Load SW data
theta_SW1 = np.load("res_SW1/theta_SW1.npy")
loss_SW1 = np.load("res_SW1/final_loss.npy")

nan_idx = np.isnan(theta_SW1).any(axis=1)
theta_SW1 = theta_SW1[~nan_idx]
loss_SW1 = loss_SW1[~nan_idx]

theta_SW1 = torch.tensor(theta_SW1, dtype=torch.float32)[:100]
print(f"theta_SW1.shape = {theta_SW1.shape}")



prop_mean = theta_SW1.mean(dim = 0, keepdims = True)
prop_std = theta_SW1.std(dim = 0, keepdims = True) 

# inflate the proposal std
prop_std *= 2

# the previous prop_std is too small for these two dimensions
prop_std[0, 0] *= 3
prop_std[0, 6] *= 3

prop_std = prop_std.clamp_min(1e-8)
print(f"Using prop_std = {prop_std}")

In [ ]:
from torch.distributions import MultivariateNormal
from torch.distributions import Gamma

# the proposal distribution
dist_prop = MultivariateNormal(loc = prop_mean.ravel().cpu(), covariance_matrix = torch.diag(prop_std.cpu().ravel()**2))
dist_prior_first8 = MultivariateNormal(loc = prior_mean.ravel().cpu(), covariance_matrix = torch.diag(prior_std.cpu().ravel()**2))
dist_prior_last4 = Gamma(concentration = prior_alpha.ravel().cpu().detach(), rate = prior_beta.ravel().cpu().detach())

### Weight for NPE
log_weight_NPE = dist_prior_first8.log_prob(theta_NPE[:, :8]) + dist_prior_last4.log_prob(torch.exp(-2 * theta_NPE[:, 8:12])).sum(dim = 1) + 4 * math.log(2) -2 * theta_NPE[:, 8:12].sum(dim = 1) - dist_prop.log_prob(theta_NPE)
weight_NPE = normalized_weight(log_weight_NPE)

### Weight for ABC
log_weight_ABC = dist_prior_first8.log_prob(theta_ABC[:, :8]) + dist_prior_last4.log_prob(torch.exp(-2 * theta_ABC[:, 8:12])).sum(dim = 1) + 4 * math.log(2) -2 * theta_ABC[:, 8:12].sum(dim = 1) - dist_prop.log_prob(theta_ABC)
weight_ABC = normalized_weight(log_weight_ABC)

In [ ]:
# resample theta_NPE with weight_NPE
theta_NPE_resampled = theta_NPE[torch.multinomial(weight_NPE, num_samples = theta_NPE.shape[0], replacement = True)]
theta_ABC_resampled = theta_ABC[torch.multinomial(weight_ABC, num_samples = theta_ABC.shape[0], replacement = True)]

In [ ]:
theta_ours.shape, theta_NPE_resampled.shape, theta_ABC_resampled.shape, theta_SW1.shape

# Bands

In [ ]:
x_simu_ours = gen_x_given_theta(theta_ours, T = 30, epis = 0.01, len_interpolate = 1/6, seed = 42, mute = True, use_soft_Ind = False)
x_simu_NPE = gen_x_given_theta(theta_NPE_resampled, T = 30, epis = 0.01, len_interpolate = 1/6, seed = 42, mute = True, use_soft_Ind = False)
x_simu_ABC = gen_x_given_theta(theta_ABC_resampled.repeat(8, 1), T = 30, epis = 0.01, len_interpolate = 1/6, seed = 42, mute = True, use_soft_Ind = False)


In [ ]:
q_low_ours = torch.quantile(x_simu_ours, 0.025, dim=0)
q_med_ours = torch.quantile(x_simu_ours, 0.5, dim=0)
q_high_ours = torch.quantile(x_simu_ours, 0.975, dim=0)

q_low_NPE = torch.quantile(x_simu_NPE, 0.025, dim=0)
q_med_NPE = torch.quantile(x_simu_NPE, 0.5, dim=0)
q_high_NPE = torch.quantile(x_simu_NPE, 0.975, dim=0)

q_low_ABC = torch.quantile(x_simu_ABC, 0.025, dim=0)
q_med_ABC = torch.quantile(x_simu_ABC, 0.5, dim=0)
q_high_ABC = torch.quantile(x_simu_ABC, 0.975, dim=0)


p = x_simu_ours.shape[1]
t = torch.linspace(0, 30, p).cpu().numpy()
x_obs_np = x_obs[:obs_size].T.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4), dpi=300, sharey=True)

plot_data = [
    (axes[0], q_low_ours, q_med_ours, q_high_ours, "Single-model"),
    (axes[1], q_low_NPE, q_med_NPE, q_high_NPE, "NPE"),
    (axes[2], q_low_ABC, q_med_ABC, q_high_ABC, "ABC"),
]

for ax, q_low, q_med, q_high, title in plot_data:
    # Plot observed time series
    ax.plot(
        t,
        x_obs_np,
        color="0.55",
        alpha=0.35,
        linewidth=0.8,
        zorder=1
    )

    # Plot posterior predictive interval
    ax.fill_between(
        t,
        q_low.cpu().numpy(),
        q_high.cpu().numpy(),
        color="tab:blue",
        alpha=0.35,
        zorder=2
    )

    # Plot posterior predictive median
    ax.plot(
        t,
        q_med.cpu().numpy(),
        color="tab:blue",
        linewidth=2.0,
        zorder=3
    )

    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Time", fontsize=16)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=14)
    ax.grid(False)

axes[0].set_ylabel(r"$y(t)$", fontsize=16)

# Create shared legend handles
obs_handle = plt.Line2D(
    [0], [0],
    color="0.55",
    alpha=0.35,
    linewidth=1.2,
    label="Observed trajectories"
)
band_handle = plt.Rectangle(
    (0, 0), 1, 1,
    color="tab:blue",
    alpha=0.35,
    label="95% posterior predictive band"
)
med_handle = plt.Line2D(
    [0], [0],
    color="tab:blue",
    linewidth=2.0,
    label="Posterior predictive median trajectory"
)

fig.legend(
    handles=[obs_handle, band_handle, med_handle],
    loc="lower center",
    ncol=3,
    frameon=False,
    fontsize=16,
    bbox_to_anchor=(0.5, -0.08)
)

fig.tight_layout(rect=[0, 0.08, 1, 1])
fig.savefig("Figs/SDE_realdata_yt.pdf", bbox_inches="tight")

plt.show()
